# LOSO Pipeline and Training - Daphnet FoG Dataset

**Objective:** Execute complete LOSO cross-validation pipeline with preprocessing and model training for FoG detection.

**Pipeline (using imblearn):**
1. Load extracted features per fold
2. Create pipeline: Imputation → Scaling → SMOTE → Classifier
3. Fit pipeline on train (automatically applies all steps)
4. Predict on test (automatically applies preprocessing, no SMOTE)
5. Aggregate results across folds

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

from imblearn.over_sampling import ADASYN
from imblearn.pipeline import Pipeline as ImbPipeline

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline

## 1. Load Feature Data

In [ ]:
# Find all fold directories
feature_dir = Path('../../outputs/daphnet_features')
fold_dirs = sorted([d for d in feature_dir.iterdir() if d.is_dir()])

print(f"Found {len(fold_dirs)} LOSO folds")
for fold_dir in fold_dirs:
    print(f"  {fold_dir.name}")

In [ ]:
# Load first fold as example
fold_0_dir = fold_dirs[0]

X_train_fold0 = pd.read_csv(fold_0_dir / 'X_train_features.csv')
y_train_fold0 = pd.read_csv(fold_0_dir / 'y_train.csv').squeeze()
X_test_fold0 = pd.read_csv(fold_0_dir / 'X_test_features.csv')
y_test_fold0 = pd.read_csv(fold_0_dir / 'y_test.csv').squeeze()

print(f"Fold 0 - {fold_0_dir.name}")
print(f"  X_train: {X_train_fold0.shape}")
print(f"  y_train: {y_train_fold0.shape} | Distribution: {np.bincount(y_train_fold0)}")
print(f"  X_test: {X_test_fold0.shape}")
print(f"  y_test: {y_test_fold0.shape} | Distribution: {np.bincount(y_test_fold0)}")
print(f"\nFeatures: {X_train_fold0.shape[1]}")

## 2. Create Pipeline

Using `imblearn.pipeline.Pipeline` with ADASYN for clean, automated preprocessing.

In [ ]:
# Create pipeline: Imputation → Scaling → ADASYN → Classifier
pipeline = ImbPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('adasyn', ADASYN(random_state=42, n_neighbors=5)),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        min_samples_split=10,
        random_state=42,
        n_jobs=-1
    ))
])

print("Pipeline created:")
for step_name, step in pipeline.steps:
    print(f"  {step_name}: {type(step).__name__}")

## 3. Train and Evaluate (Single Fold)

The pipeline automatically handles all preprocessing steps.

In [ ]:
# Fit pipeline on train data (all steps applied automatically)
print("Training pipeline on Fold 0...")
pipeline.fit(X_train_fold0, y_train_fold0)
print("Training complete")

# Predict on test data (preprocessing applied, SMOTE skipped automatically)
y_pred = pipeline.predict(X_test_fold0)

# Evaluate
accuracy = accuracy_score(y_test_fold0, y_pred)
precision = precision_score(y_test_fold0, y_pred, average='binary')
recall = recall_score(y_test_fold0, y_pred, average='binary')
f1 = f1_score(y_test_fold0, y_pred, average='binary')

print(f"\nFold 0 Results:")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1-Score:  {f1:.4f}")

In [ ]:
# Visualize ADASYN effect
# Access the resampled data from ADASYN step
adasyn_step = pipeline.named_steps['adasyn']

# Transform data up to ADASYN step to see effect
X_imputed = pipeline.named_steps['imputer'].transform(X_train_fold0)
X_scaled = pipeline.named_steps['scaler'].transform(X_imputed)
X_resampled, y_resampled = adasyn_step.fit_resample(X_scaled, y_train_fold0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Before ADASYN
original_dist = np.bincount(y_train_fold0)
axes[0].bar(['No FoG', 'FoG'], original_dist, color=['steelblue', 'coral'])
axes[0].set_ylabel('Sample Count')
axes[0].set_title('Before ADASYN (Train)', fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(original_dist):
    axes[0].text(i, v, f'{v:,}', ha='center', va='bottom')

# After ADASYN
resampled_dist = np.bincount(y_resampled)
axes[1].bar(['No FoG', 'FoG'], resampled_dist, color=['steelblue', 'coral'])
axes[1].set_ylabel('Sample Count')
axes[1].set_title('After ADASYN (Train)', fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(resampled_dist):
    axes[1].text(i, v, f'{v:,}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print(f"\nOriginal: {len(y_train_fold0):,} samples")
print(f"After ADASYN: {len(y_resampled):,} samples")
print(f"\nADASYN focuses on harder-to-learn examples near decision boundaries")

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test_fold0, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
           xticklabels=['No FoG', 'FoG'],
           yticklabels=['No FoG', 'FoG'],
           cbar_kws={'label': 'Count'})
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix - Fold 0\nAccuracy: {accuracy:.4f}', fontweight='bold')
plt.tight_layout()
plt.show()

# Classification report
print("\nClassification Report:")
print(classification_report(y_test_fold0, y_pred, target_names=['No FoG', 'FoG']))

## 4. Complete LOSO Pipeline

Run the full pipeline on all folds using clean, concise code.

In [ ]:
# Run LOSO pipeline
print("Running complete LOSO pipeline...")
print("="*70)

results = []

for fold_dir in tqdm(fold_dirs, desc="Processing folds"):
    fold_name = fold_dir.name
    
    # Load data
    X_train = pd.read_csv(fold_dir / 'X_train_features.csv')
    y_train = pd.read_csv(fold_dir / 'y_train.csv').squeeze()
    X_test = pd.read_csv(fold_dir / 'X_test_features.csv')
    y_test = pd.read_csv(fold_dir / 'y_test.csv').squeeze()
    
    # Create and fit pipeline
    pipeline = ImbPipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('adasyn', ADASYN(random_state=42, n_neighbors=5)),
        ('classifier', RandomForestClassifier(
            n_estimators=100,
            max_depth=20,
            min_samples_split=10,
            random_state=42,
            n_jobs=-1
        ))
    ])
    
    pipeline.fit(X_train, y_train)
    
    # Predict and evaluate
    y_pred = pipeline.predict(X_test)
    
    results.append({
        'fold': fold_name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='binary', zero_division=0),
        'recall': recall_score(y_test, y_pred, average='binary', zero_division=0),
        'f1': f1_score(y_test, y_pred, average='binary', zero_division=0),
        'cm': confusion_matrix(y_test, y_pred),
        'n_train': len(y_train),
        'n_test': len(y_test)
    })

print("\n" + "="*70)
print("LOSO pipeline complete")
print("="*70)

## 5. Results Analysis

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame([
    {k: v for k, v in r.items() if k != 'cm'}
    for r in results
])

print("Per-Fold Results:")
print(results_df.to_string(index=False))

# Aggregate metrics
print("\n" + "="*70)
print("AGGREGATE METRICS (mean ± std)")
print("="*70)
print(f"Accuracy:  {results_df['accuracy'].mean():.4f} ± {results_df['accuracy'].std():.4f}")
print(f"Precision: {results_df['precision'].mean():.4f} ± {results_df['precision'].std():.4f}")
print(f"Recall:    {results_df['recall'].mean():.4f} ± {results_df['recall'].std():.4f}")
print(f"F1-Score:  {results_df['f1'].mean():.4f} ± {results_df['f1'].std():.4f}")
print("="*70)

In [ ]:
# Visualize results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = ['accuracy', 'precision', 'recall', 'f1']
titles = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = ['steelblue', 'coral', 'gold', 'lightgreen']

for idx, (metric, title, color) in enumerate(zip(metrics, titles, colors)):
    ax = axes[idx // 2, idx % 2]
    
    values = results_df[metric].values
    x = np.arange(len(values))
    
    ax.bar(x, values, color=color, alpha=0.7)
    ax.axhline(y=values.mean(), color='red', linestyle='--', 
              label=f'Mean: {values.mean():.3f}')
    ax.set_xlabel('Fold')
    ax.set_ylabel(title)
    ax.set_title(f'{title} per Fold', fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([f"F{i}" for i in range(len(values))], fontsize=9)
    ax.set_ylim([0, 1.05])
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Aggregate confusion matrix
cm_total = np.sum([r['cm'] for r in results], axis=0)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_total, annot=True, fmt='d', cmap='Blues',
           xticklabels=['No FoG', 'FoG'],
           yticklabels=['No FoG', 'FoG'],
           cbar_kws={'label': 'Count'})
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Aggregated Confusion Matrix (All Folds)', fontweight='bold')
plt.tight_layout()
plt.show()

# Compute normalized confusion matrix
cm_normalized = cm_total.astype('float') / cm_total.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues',
           xticklabels=['No FoG', 'FoG'],
           yticklabels=['No FoG', 'FoG'],
           cbar_kws={'label': 'Percentage'})
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Normalized Confusion Matrix (All Folds)', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Save Results

In [ ]:
# Save results
output_dir = Path('../../outputs')
output_dir.mkdir(parents=True, exist_ok=True)

results_path = output_dir / 'loso_results_binary.csv'
results_df.to_csv(results_path, index=False)

print(f"Results saved: {results_path}")
print(f"\nSummary:")
print(f"  Folds: {len(results_df)}")
print(f"  Mean Accuracy: {results_df['accuracy'].mean():.4f}")
print(f"  Mean F1-Score: {results_df['f1'].mean():.4f}")

## Summary

**Completed LOSO pipeline using imblearn with ADASYN:**

1. Loaded extracted features from 10 LOSO folds
2. Created pipeline with 4 steps:
   - SimpleImputer (median strategy)
   - StandardScaler (fit on train)
   - ADASYN (applied ONLY on train, adaptive synthetic sampling)
   - RandomForestClassifier
3. Fit pipeline on train (all steps applied automatically)
4. Predict on test (preprocessing applied, ADASYN skipped)
5. Aggregated results across all folds

**Why ADASYN instead of SMOTE:**
- ADASYN focuses on harder-to-learn examples near decision boundaries
- More appropriate for time-series data than standard SMOTE
- Reduces risk of creating unrealistic synthetic examples
- Better handles class imbalance in sequential data

**Advantages of imblearn pipeline:**
- Clean, concise code (3 lines vs 10+ lines per fold)
- Automatic fit/transform handling
- No risk of data leakage
- ADASYN correctly applied only during training
- Easy to modify or extend pipeline

**Performance metrics:**
- Mean accuracy, precision, recall, and F1-score computed across folds
- Confusion matrices show true/false positive/negative rates
- Results indicate model capability for FoG detection

**Key improvements implemented:**
- Test windows with 0% overlap (realistic evaluation)
- ADASYN for better synthetic sample generation
- Focus on F1 and Recall as primary metrics for imbalanced data

**Next steps:**
- Hyperparameter tuning using GridSearchCV with the pipeline
- Try different classifiers (SVM, XGBoost, Neural Networks)
- Add feature selection step to pipeline
- Ensemble methods for improved performance